In [ ]:
Import Library
import pandas as pd
import numpy as np
import regex as re
import pandas as pd
import numpy as np
import regex as re
import pandas as pd
from factor_analyzer.factor_analyzer import calculate_bartlett_sphericity, FactorAnalyzer
import matplotlib.pyplot as plt
from scipy.stats.stats import pearsonr
from factor_analyzer.factor_analyzer import calculate_kmo

df = pd.read_excel('Final Data.xlsx') #This is not public access in lines with ethics permissions and gdpr

/usr/local/lib/python3.12/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [ ]:
#Drop Prolific Header
header = df.iloc[0]
df.drop([0], inplace = True)

#Remove bot-completion
df.drop([209], inplace = True)

In [ ]:
#Need to ensure even split of categorical and numerical variables between EFA and CFA
cat_count = []
num_count = []

for index, row in df.iterrows():
  # categorical_nums = row[[cat_cols]]
  cat_count.append(sum(row[['Q4b_1', 'Q5b_1', 'Q6b_1', 'Q7b_1']].notna()))
  num_count.append(sum(row[['Q8b_1', 'Q9b_1', 'Q10b_1', 'Q11b_1', 'Q12b_1']].notna()))

In [ ]:
from sklearn.model_selection import train_test_split

EFA, CFA = train_test_split(df, test_size = 0.5, stratify = num_count)
CFA.shape

(720, 220)

In [ ]:
EFA.to_excel('/content/EFA Data.xlsx')
CFA.to_excel('/content/CFA Data.xlsx')

In [ ]:
cat_count = []
num_count = []

for index, row in EFA.iterrows():
  # categorical_nums = row[[cat_cols]]
  cat_count.append(sum(row[['Q4b_2', 'Q5b_2', 'Q6b_2', 'Q7b_2']].notna()))
  num_count.append(sum(row[['Q8b_1', 'Q9b_1', 'Q10b_1', 'Q11b_1', 'Q12b_1']].notna()))

In [ ]:
print(sum(cat_count))
print(sum(num_count))

953
1207


In [ ]:
import pandas as pd
a = ['CFA', 'EFA']
b = ['cat', 'num']

for x in a:
  for y in b:
    df = pd.read_csv('/content/' + x + '_' + y + '.csv')
    df = df.sample(frac=1).reset_index(drop=True).drop('Unnamed: 0', axis = 1)
    df.to_csv('/content/' + x + '_' + y + '_final.csv')

#Preprocessing Script
This process is the same of EFA and CFA

In [ ]:
df = pd.read_excel('/content/EFA Data.xlsx', index_col = 0)

In [ ]:
#First check if anyone actually failed the attention check

def check_failed_attention_checks(dataframe):
  """Check if there are any failed attention checks"""
  failed_items = []
  for x in range(4, 13):
    if x < 8:
      attention_check_col = 21
    else:
      attention_check_col = 23

    failed_checks = [x for x in dataframe[dataframe['Q' + str(x) + 'b_' + str(attention_check_col)].notnull()]['Q' + str(x) + 'b_' + str(attention_check_col)] if x != 'Neutral']


    if len(failed_checks) > 1:
      failed_items.append(f"There are {len(failed_checks)} failed attention checks for item {x}")
    else:
      continue

  if len(failed_items) == 0:
    return 'There are no failed attention checks'

  else:
    return failed_items

In [ ]:
#Let's remove failed attention checks item-wise

def remove_failed_attention(dataframe, columns):
  """Sets feature ratings with failed attention checks to NaN"""
  #Drop header column
  # dataframe = dataframe.drop(0, axis = 'index')

  #First get index of failed attention checks
  for col in columns:
    indices_failed = [x for x in dataframe[dataframe[col].notnull()][col].index.to_list() if dataframe.loc[x, col] != 'Neutral']

    print(indices_failed)

    if len(indices_failed) > 0: #if anyone failed attention check set their answer to 0

      #Identify columns relating to question
      if columns.index(col) in range(0, 4):
        question_columns = ['Q' + str(columns.index(col) + 4) + 'b_' + str(y) for y in range(1, 22)]
      else:
        question_columns = ['Q' + str(columns.index(col) + 4) + 'b_' + str(y) for y in range(1, 24)]

      for ind in indices_failed:
        dataframe.loc[ind, question_columns] = np.nan

    else: #no failure: move on to next column
      continue

  return dataframe

In [ ]:
#First see if there are any failed attention checks:

attention_columns = ['Q4b_21', 'Q5b_21', 'Q6b_21', 'Q7b_21', 'Q8b_23', 'Q9b_23', 'Q10b_23', 'Q11b_23', 'Q12b_23']

fail_len = check_failed_attention_checks(df)
print(fail_len)

In [ ]:
df = remove_failed_attention(df, attention_columns)

In [ ]:
check_failed_attention_checks(df)

In [ ]:
df['Duration (in seconds)'] = pd.to_numeric(df['Duration (in seconds)'], errors='coerce')

df.boxplot('Duration (in seconds)')
#Shows no excessively fast responses. Slow responses are retained.

In [ ]:
#Isolate only item-answers
df.columns.to_list().index('Q4b_1')

answer_columns = df.columns.to_list()[df.columns.to_list().index('Q4b_1') : df.columns.to_list().index('Q24')]

answer_df = df[answer_columns]

In [ ]:
#Let's code the strongly disagree to strongly agree

value_dict = {'Strongly Disagree' : 1, 'Disagree' : 2, 'Neutral' : 3, 'Agree' : 4, 'Strongly Agree' : 5}

answer_df.replace(value_dict, inplace=True)

#Categorical Analysis

In [ ]:
#Q4 - Q7 are Categorical
cb = 'Q4b_1'
ce = 'Q8b_1'

categorical_df = answer_df[answer_df.columns.to_list()[:answer_df.columns.to_list().index(ce)]]

#Write a little function to split by Column Index and Stack by question
#function needs to take number of features (4 for categorical), number of items (21 for categorical),
#Needs to return dataframe
import numpy as np

def restructure_dataframe(n_items, dataframe):
  """Takes dataframe and re-stacks it by item"""
  #Iterate through features
  for x in range(4, 8):
    col_names = []
    for y in range(1, n_items + 1):
      col_names.append('Q' + str(x) + 'b_' + str(y))

    if x == 4:
      new_dataframe = dataframe[col_names]
    else:
      new_dataframe = np.vstack([new_dataframe, dataframe[col_names]])

  new_dataframe = pd.DataFrame(new_dataframe).dropna()

  new_dataframe = new_dataframe.drop(20, axis = 1)

  return new_dataframe

re_categorical_df = restructure_dataframe(21, categorical_df)

re_categorical_df.to_csv('/content/EFA_cat.csv') #This file is accessible on GitHub -> From here go to R-based analysis

#Numerical Analysis

In [ ]:
#Q4 - Q7 are Numerical
cb = 'Q8b_1'

numerical_df = answer_df[answer_df.columns.to_list()[answer_df.columns.to_list().index(cb):]]

#Write a little function to split by Column Index and Stack by question
#function needs to take number of features (4 for categorical), number of items (21 for categorical),
#Needs to return dataframe
import numpy as np

def restructure_dataframe(n_features, n_items, dataframe):
  """Takes dataframe and re-stacks it by item"""
  #Iterate through features
  for x in range(8, 13):
    col_names = []
    for y in range(1, n_items + 1):
      col_names.append('Q' + str(x) + 'b_' + str(y))

    if x == 8:
      new_dataframe = dataframe[col_names]
    else:
      new_dataframe = np.vstack([new_dataframe, dataframe[col_names]])


  new_dataframe = pd.DataFrame(new_dataframe).dropna()

  new_dataframe = new_dataframe.drop(22, axis = 1)

  return new_dataframe

re_numerical_df = restructure_dataframe(5, 23, numerical_df)

re_numerical_df.to_csv('/content/EFA_num.csv') #This is accessible on Github - From here R-based analysis is used.